In [ ]:

import os
import time
import warnings
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"🖥️ Устройство: {DEVICE}")

class FrequencyBranch(nn.Module):
    """Частотная ветка для анализа GAN-артефактов"""
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1)
        )
        
    def forward(self, x):
        fft = torch.fft.fft2(x, norm='ortho')
        amp = torch.abs(fft)
        amp = torch.log1p(amp)
        # Нормализация
        amp = (amp - amp.mean(dim=[2,3], keepdim=True)) / (amp.std(dim=[2,3], keepdim=True) + 1e-6)
        return self.conv(amp).flatten(1)


class HighFrequencyBranch(nn.Module):
    """Высокочастотная ветка (разность с размытием)"""
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1)
        )
        
    def forward(self, x):
        blurred = F.avg_pool2d(x, 3, stride=1, padding=1)
        hf = x - blurred
        return self.conv(hf).flatten(1)


class RGBBranch(nn.Module):
    """Основная RGB ветка"""
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1)
        )
        
    def forward(self, x):
        return self.conv(x).flatten(1)


class LocalArtifactBranch(nn.Module):
    """Анализ локальных артефактов через патчи"""
    def __init__(self, patch_size=32, stride=16):
        super().__init__()
        self.patch_conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=patch_size, stride=stride, padding=0),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x):
        feat = self.patch_conv(x).flatten(1)
        return self.fc(feat)


class GANDetector(nn.Module):
    """
    Детектор GAN-артефактов
    """
    def __init__(self, num_classes=1):
        super().__init__()
        
        self.rgb_branch = RGBBranch()
        self.freq_branch = FrequencyBranch()
        self.hf_branch = HighFrequencyBranch()
        self.artifact_branch = LocalArtifactBranch(patch_size=32, stride=16)
        
        total_dim = 256 + 128 + 64 + 128
        
        # Классификатор
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(total_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        rgb_feat = self.rgb_branch(x)
        freq_feat = self.freq_branch(x)
        hf_feat = self.hf_branch(x)
        artifact_feat = self.artifact_branch(x)
        
        combined = torch.cat([rgb_feat, freq_feat, hf_feat, artifact_feat], dim=1)
        logits = self.classifier(combined).squeeze(1)
        
        return logits


train_transform = T.Compose([
    T.Resize((256, 256)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=5),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

val_transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])


class FaceDataset(Dataset):
    def __init__(self, img_dir, filenames, labels, transform=None):
        self.transform = transform
        self.img_dir = Path(img_dir)
        self.paths = []
        
        for name in filenames:
            name = str(name)
            for ext in ['.jpg', '.png', '.jpeg']:
                p = self.img_dir / (name + ext)
                if p.exists():
                    self.paths.append(p)
                    break
        
        self.labels = torch.tensor(labels, dtype=torch.float32)
        print(f"   Найдено {len(self.paths)} из {len(filenames)} файлов")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


class TestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = Path(img_dir)
        self.paths = sorted([p for p in self.img_dir.iterdir() 
                            if p.suffix.lower() in ['.jpg', '.jpeg', '.png']])
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, idx


class FocalLoss(nn.Module):
    """Focal Loss для борьбы с дисбалансом классов"""
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        
    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        
        focal_weight = (1 - pt) ** self.gamma
        
        alpha_weight = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        
        loss = alpha_weight * focal_weight * bce_loss
        return loss.mean()


def prepare_loaders(data_dir, batch_size, val_split=0.15):
    csv_path = os.path.join(data_dir, 'train_solution.csv')
    df = pd.read_csv(csv_path)
    fnames = df.iloc[:, 0].astype(str).tolist()
    labels = df.iloc[:, 1].values.astype(np.float32)
    
    skf = StratifiedKFold(n_splits=int(1/val_split), shuffle=True, random_state=42)
    train_idx, val_idx = next(skf.split(fnames, labels))
    
    train_ds = FaceDataset(
        os.path.join(data_dir, 'train_images'),
        [fnames[i] for i in train_idx],
        labels[train_idx],
        transform=train_transform
    )
    val_ds = FaceDataset(
        os.path.join(data_dir, 'train_images'),
        [fnames[i] for i in val_idx],
        labels[val_idx],
        transform=val_transform
    )
    
    train_labels = labels[train_idx]
    class_counts = np.bincount(train_labels.astype(int))
    class_weights = 1.0 / (class_counts + 1e-8)
    sample_weights = class_weights[train_labels.astype(int)]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
    
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, sampler=sampler,
        num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=True
    )
    
    return train_loader, val_loader


def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0
    
    for imgs, labels in tqdm(loader, desc="Train", leave=False):
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)


def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Val", leave=False):
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            
            with autocast():
                logits = model(imgs)
                loss = criterion(logits, labels)
                probs = torch.sigmoid(logits)
            
            total_loss += loss.item()
            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())
    
    all_probs = torch.cat(all_probs)
    all_labels = torch.cat(all_labels)
    
    best_f1 = 0
    best_th = 0.5
    
    for th in np.linspace(0.2, 0.85, 50):
        preds = (all_probs >= th).float()
        f1 = f1_score(all_labels.numpy(), preds.numpy())
        if f1 > best_f1:
            best_f1 = f1
            best_th = th
    
    return total_loss / len(loader), best_f1, best_th


def train_model(data_dir='./dataset', epochs=50, batch_size=32, lr=1e-3):
    print("\n" + "="*70)
    print("🔬 GAN DETECTOR — ДЕТЕКЦИЯ СГЕНЕРИРОВАННЫХ ЛИЦ")
    print("="*70)
    
    train_loader, val_loader = prepare_loaders(data_dir, batch_size)
    print(f"✅ Train: {len(train_loader)} батчей, Val: {len(val_loader)} батчей")
    
    model = GANDetector(num_classes=1)
    
    if GPU_COUNT > 1:
        print(f"🚀 Использование {GPU_COUNT} GPU")
        model = nn.DataParallel(model)
        lr = lr * GPU_COUNT
    
    model = model.to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"📊 Параметров: {total_params:,}")
    
    criterion = FocalLoss(alpha=0.75, gamma=2.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )
    scaler = GradScaler() if DEVICE.type == 'cuda' else None
    
    history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
    best_f1 = 0.0
    best_threshold = 0.5
    
    for epoch in range(epochs):
        start_time = time.time()
        
        train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler)
        val_loss, val_f1, val_th = validate(model, val_loader, criterion)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        
        scheduler.step()
        
        epoch_time = time.time() - start_time
        current_lr = scheduler.get_last_lr()[0]
        print(f"\n📊 Epoch {epoch+1}/{epochs} | Time: {epoch_time:.1f}s | LR: {current_lr:.2e}")
        print(f"   Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_threshold = val_th
            # Сохраняем на CPU
            model_to_save = model.module if GPU_COUNT > 1 else model
            torch.save(model_to_save.state_dict(), 'best_gan_detector.pth')
            print(f"   💾 Новый лучший F1: {best_f1:.4f} с порогом {best_threshold:.3f}")
        
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()
    
    print("\n" + "="*70)
    print(f"✅ ОБУЧЕНИЕ ЗАВЕРШЕНО!")
    print(f"🏆 Лучший F1: {best_f1:.4f} с порогом {best_threshold:.3f}")
    print("="*70)
    
    return best_threshold
def inference(model_path='best_gan_detector.pth', 
              test_dir='./dataset/test_images',
              batch_size=64,
              threshold=0.5):
    
    print("\n" + "="*70)
    print("🔮 ЗАПУСК ИНФЕРЕНСА")
    print("="*70)
    
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Модель не найдена: {model_path}")
    
    test_transform = T.Compose([
        T.Resize((256, 256)),
        T.ToTensor(),
        T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])
    
    test_dataset = TestDataset(test_dir, transform=test_transform)
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )
    
    print(f"📁 Найдено изображений: {len(test_dataset)}")
    
    model = GANDetector(num_classes=1)
    state_dict = torch.load(model_path, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model = model.to(DEVICE)
    model.eval()
    
    print(f"✅ Модель загружена")
    print(f"🎯 Порог: {threshold}")
    
    all_preds = []
    all_indices = []
    
    with torch.no_grad():
        for imgs, indices in tqdm(test_loader, desc="Инференс"):
            imgs = imgs.to(DEVICE, non_blocking=True)
            
            with autocast():
                logits = model(imgs)
                probs = torch.sigmoid(logits)
                preds = (probs >= threshold).int()
            
            all_preds.extend(preds.cpu().numpy())
            all_indices.extend(indices.numpy())
    
    sorted_pairs = sorted(zip(all_indices, all_preds), key=lambda x: x[0])
    final_preds = [pred for _, pred in sorted_pairs]
    
    return final_preds


def create_submission(predictions, output_file='submission.csv'):
    df = pd.DataFrame({
        'Id': range(len(predictions)),
        'target_feature': predictions
    })
    df.to_csv(output_file, index=False)
    print(f"\n✅ Файл решения сохранён: {output_file}")
    
    pos_count = sum(predictions)
    print(f"📊 Предсказано класса 1: {pos_count} ({pos_count/len(predictions):.1%})")
    
    return df
if __name__ == '__main__':
    DATA_DIR = './dataset'
    
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    
    best_th = train_model(
        data_dir=DATA_DIR,
        epochs=50,
        batch_size=32,
        lr=1e-3
    )
    
    predictions = inference(
        model_path='best_gan_detector.pth',
        test_dir=f'{DATA_DIR}/test_images',
        batch_size=64,
        threshold=best_th
    )
    
    create_submission(predictions, 'submission.csv')
    
    print("\n🎉 РАБОТА ЗАВЕРШЕНА!")

🖥️ Устройство: cuda

🔬 GAN DETECTOR — ДЕТЕКЦИЯ СГЕНЕРИРОВАННЫХ ЛИЦ
   Найдено 41665 из 41665 файлов
   Найдено 8334 из 8334 файлов
✅ Train: 1303 батчей, Val: 261 батчей
📊 Параметров: 1,087,777



📊 Epoch 1/50 | Time: 673.0s | LR: 9.76e-04
   Train Loss: 0.0709 | Val Loss: 0.0791 | Val F1: 0.3441
   💾 Новый лучший F1: 0.3441 с порогом 0.598



📊 Epoch 2/50 | Time: 377.0s | LR: 9.05e-04
   Train Loss: 0.0665 | Val Loss: 0.0620 | Val F1: 0.4253
   💾 Новый лучший F1: 0.4253 с порогом 0.598



📊 Epoch 3/50 | Time: 302.1s | LR: 7.94e-04
   Train Loss: 0.0635 | Val Loss: 0.0746 | Val F1: 0.4633
   💾 Новый лучший F1: 0.4633 с порогом 0.638



📊 Epoch 4/50 | Time: 266.9s | LR: 6.55e-04
   Train Loss: 0.0611 | Val Loss: 0.0584 | Val F1: 0.4660
   💾 Новый лучший F1: 0.4660 с порогом 0.624


...


📊 Epoch 48/50 | Time: 200.8s | LR: 5.79e-04
   Train Loss: 0.0365 | Val Loss: 0.0344 | Val F1: 0.6746



📊 Epoch 49/50 | Time: 213.7s | LR: 5.40e-04
   Train Loss: 0.0373 | Val Loss: 0.0365 | Val F1: 0.6745



📊 Epoch 50/50 | Time: 201.3s | LR: 5.01e-04
   Train Loss: 0.0356 | Val Loss: 0.0369 | Val F1: 0.7003
   💾 Новый лучший F1: 0.7003 с порогом 0.651

✅ ОБУЧЕНИЕ ЗАВЕРШЕНО!
🏆 Лучший F1: 0.7003 с порогом 0.651

🔮 ЗАПУСК ИНФЕРЕНСА
📁 Найдено изображений: 10000
✅ Модель загружена
🎯 Порог: 0.6510204081632652


Инференс: 100%|██████████| 157/157 [01:37<00:00,  1.62it/s]


✅ Файл решения сохранён: submission.csv
📊 Предсказано класса 1: 1714 (17.1%)

🎉 РАБОТА ЗАВЕРШЕНА!
